# CareSync Calendar Dimension

Builds the standard calendar table `adb_caresync.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31.

In [0]:
%sql
-- Standard calendar dimension: one row per day, 1940-01-01 to 2030-12-31
CREATE OR REPLACE TABLE adb_caresync.silver.calendar
COMMENT 'Standard calendar dimension (1940-01-01 to 2030-12-31)'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)                 AS datekey,
  d                                                       AS date,
  -- Year
  year(d)                                                 AS year,
  CASE WHEN (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0
       THEN true ELSE false END                           AS is_leap_year,
  CAST(date_trunc('YEAR', d) AS DATE)                     AS first_day_of_year,
  make_date(year(d), 12, 31)                              AS last_day_of_year,
  -- Half year
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END               AS half_year,
  concat('H', CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END)  AS half_year_name,
  -- Quarter
  quarter(d)                                              AS quarter,
  concat('Q', quarter(d))                                 AS quarter_name,
  concat(year(d), '-Q', quarter(d))                       AS year_quarter,
  CAST(date_trunc('QUARTER', d) AS DATE)                  AS first_day_of_quarter,
  date_sub(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), 1) AS last_day_of_quarter,
  datediff(d, CAST(date_trunc('QUARTER', d) AS DATE)) + 1 AS day_of_quarter,
  -- Month
  month(d)                                                AS month,
  date_format(d, 'MMMM')                                  AS month_name,
  date_format(d, 'MMM')                                   AS month_short_name,
  date_format(d, 'yyyy-MM')                               AS year_month,
  CAST(date_format(d, 'yyyyMM') AS INT)                   AS year_month_key,
  CAST(date_trunc('MONTH', d) AS DATE)                    AS first_day_of_month,
  last_day(d)                                             AS last_day_of_month,
  day(last_day(d))                                        AS days_in_month,
  -- Week (ISO 8601: weeks start Monday)
  weekofyear(d)                                           AS week_of_year,
  extract(YEAROFWEEK FROM d)                              AS iso_year,
  CAST(ceil(day(d) / 7.0) AS INT)                         AS week_of_month,
  CAST(date_trunc('WEEK', d) AS DATE)                     AS week_start_date,
  date_add(CAST(date_trunc('WEEK', d) AS DATE), 6)        AS week_end_date,
  -- Day
  day(d)                                                  AS day_of_month,
  dayofyear(d)                                            AS day_of_year,
  weekday(d) + 1                                          AS day_of_week,       -- 1 = Monday ... 7 = Sunday
  date_format(d, 'EEEE')                                  AS day_name,
  date_format(d, 'EEE')                                   AS day_short_name,
  -- Flags
  weekday(d) >= 5                                         AS is_weekend,
  weekday(d) < 5                                          AS is_weekday,
  d = CAST(date_trunc('MONTH', d) AS DATE)                AS is_month_start,
  d = last_day(d)                                         AS is_month_end,
  d = CAST(date_trunc('QUARTER', d) AS DATE)              AS is_quarter_start,
  d = date_sub(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), 1) AS is_quarter_end,
  d = CAST(date_trunc('YEAR', d) AS DATE)                 AS is_year_start,
  d = make_date(year(d), 12, 31)                          AS is_year_end
FROM dates
ORDER BY d

In [0]:
%sql
-- Validation: expected 33,238 rows (91 years x 365 + 23 leap days), unique datekeys
SELECT
  COUNT(*)                 AS row_count,
  COUNT(DISTINCT datekey)  AS distinct_datekeys,
  MIN(date)                AS min_date,
  MAX(date)                AS max_date,
  SUM(CASE WHEN is_leap_year AND month = 2 AND day_of_month = 29 THEN 1 ELSE 0 END) AS leap_days
FROM adb_caresync.silver.calendar

In [0]:
display(_sqldf)

In [0]:
%sql
SELECT * FROM adb_caresync.silver.calendar WHERE date BETWEEN DATE'2026-12-28' AND DATE'2027-01-04' ORDER BY date